<a href="https://colab.research.google.com/github/janani26121992/AI-Projects/blob/main/LSTM_text_gen_on_Shakespeare_data_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Generation Model**
# 1. IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# **Data Gathering**

In [4]:
with open("/content/Harry Potter and the Sorcerer's Sto.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:500])

Harry Potter and the Sorcerer's Stone 

CHAPTER ONE 

THE BOY WHO LIVED 

Mr. and Mrs. Dursley, of number four, Privet Drive, were proud to say that they were perfectly normal, thank you very much. They were the last people you'd expect to be involved in anything strange or mysterious, because they just didn't hold with such nonsense. 

Mr. Dursley was the director of a firm called Grunnings, which made drills. He was a big, beefy man with hardly any neck, although he did have a very large musta


In [5]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items()}

Vocabulary Size: 6032


In [6]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'a': 4,
 'he': 5,
 'of': 6,
 'harry': 7,
 'was': 8,
 'it': 9,
 'in': 10,
 'his': 11,
 'you': 12,
 'said': 13,
 'had': 14,
 'i': 15,
 'on': 16,
 'at': 17,
 'that': 18,
 'they': 19,
 'as': 20,
 'him': 21,
 'but': 22,
 'with': 23,
 'ron': 24,
 'all': 25,
 'out': 26,
 'for': 27,
 'up': 28,
 'be': 29,
 'what': 30,
 'hagrid': 31,
 'them': 32,
 'were': 33,
 'have': 34,
 'there': 35,
 'back': 36,
 'hermione': 37,
 'one': 38,
 'this': 39,
 'if': 40,
 'from': 41,
 'so': 42,
 'not': 43,
 'she': 44,
 'about': 45,
 'into': 46,
 'me': 47,
 'their': 48,
 'know': 49,
 'been': 50,
 'off': 51,
 'got': 52,
 'no': 53,
 'could': 54,
 "didn't": 55,
 'like': 56,
 'get': 57,
 'down': 58,
 'professor': 59,
 'just': 60,
 'her': 61,
 'see': 62,
 'who': 63,
 'when': 64,
 'is': 65,
 'very': 66,
 'over': 67,
 'looked': 68,
 'do': 69,
 'now': 70,
 'then': 71,
 'are': 72,
 'we': 73,
 'by': 74,
 "it's": 75,
 'your': 76,
 "don't": 77,
 'snape': 78,
 'around': 79,
 'dumbledore': 80,
 "he

## **CREATE INPUT SEQUENCES : **
Instead of splitting line by line, use continuous text for better learning

In [7]:
token_list = tokenizer.texts_to_sequences([text])[0]

input_sequences = []

# Create sequences of 6 words
# First 5 words = input, 6th word = target
for i in range(5, len(token_list)):
    n_gram_sequence = token_list[i-5:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[7, 121, 2, 1, 634, 158], [121, 2, 1, 634, 158, 611], [2, 1, 634, 158, 611, 38], [1, 634, 158, 611, 38, 1], [634, 158, 611, 38, 1, 144]]


# **PADDING**
# **SPLIT X AND y**

In [8]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

'''
[7, 121, 2, 1, 634, 158]
x= [7, 121, 2, 1, 634 ]
y = [158]
'''

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (77920, 5)
y shape: (77920, 6032)


# **BUILD LSTM MODEL**

In [9]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       772,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6032)           │       609,232 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,659,228 (6.33 MB)

 Trainable params: 1,659,228 (6.33 MB)

 Non-trainable params: 0 (0.00 B)

# **Model training**

In [10]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])


Epoch 1/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 91s 36ms/step - accuracy: 0.0479 - loss: 6.6433
Epoch 2/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 84s 34ms/step - accuracy: 0.0645 - loss: 6.1727
Epoch 3/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 83s 34ms/step - accuracy: 0.0874 - loss: 5.8483
Epoch 4/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 82s 34ms/step - accuracy: 0.1023 - loss: 5.5951
Epoch 5/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 145s 35ms/step - accuracy: 0.1100 - loss: 5.3972
Epoch 6/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 85s 35ms/step - accuracy: 0.1169 - loss: 5.2242
Epoch 7/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 142s 35ms/step - accuracy: 0.1249 - loss: 5.0700
Epoch 8/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 141s 35ms/step - accuracy: 0.1318 - loss: 4.9259
Epoch 9/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 142s 35ms/step - accuracy: 0.1385 - loss: 4.7972
Epoch 10/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 83s 34ms/step - accuracy: 0.1451 - loss: 4.6777
Epoch 11/30
2435/2435 ━━━━━━━━━━━━━━━━━━━━ 143s 34ms/step - accuracy: 0.1509 - loss: 4.5658
Epo

# Using the trained model for final predictions
TEMPERATURE + TOP-K SAMPLING
Temperature effect
0.5 → safer predictions
0.8 → balanced
1.2 → more creative/random

In [11]:
model.save("TextGenerationModel.keras")

In [12]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

# **Text Generation method**

In [13]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        ) # [0,0,0,189,45]

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

# **Predictions**

In [14]:
print(generate_text("So shaken", next_words=15))

So shaken his nose he was the most shaped clothes and clinking he had a very small
